# Introduction

The purpose of this notebook is to build a chain that:
1. Uses Messages
2. Uses Chat models in graph nodes
3. Uses Tools binded to chat models
4. Execute tool calls in graph nodes

# Import

In [1]:
import tomllib
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage #SystemMessage, BaseMessage

c:\Users\ssrim\anaconda3\envs\lc-academy-env\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


# Part 1: Messages

In this section, we learn what the various Message classes are (currently will limit to HumanMessage and AIMessage)

In [2]:
# Loading secrets to get API key

with open("../secrets.toml", "rb") as f:
    secrets = tomllib.load(f)

In [3]:
# Creating a message object

m = HumanMessage(content="What is the capital of France?")

m.pretty_print()

================================ Human Message =================================

What is the capital of France?


In [4]:
# Creating a conversation for a Chat Model

messages = [
    HumanMessage(content="Do you know what MCP and A2A protocols are used for?", name="User1"),
    AIMessage(content="Yes, they are used for different purposes when working with LLMs.", name="Model"),
    HumanMessage(content="Are they used for the same purpose?", name="User1"),
    AIMessage(content="No, they have distinct use cases.", name="Model"),
    HumanMessage(content="Can you explain the difference with short and sweet examples?", name="User1"),
]

for m in messages:
    m.pretty_print()

================================ Human Message =================================
Name: User1

Do you know what MCP and A2A protocols are used for?
================================== Ai Message ==================================
Name: Model

Yes, they are used for different purposes when working with LLMs.
================================ Human Message =================================
Name: User1

Are they used for the same purpose?
================================== Ai Message ==================================
Name: Model

No, they have distinct use cases.
================================ Human Message =================================
Name: User1

Can you explain the difference with short and sweet examples?


# Part 2: Chat Models

Note: There is a nuanced difference between LLM models and Chat models. LLM models accept a user query but Chat models can accept a whole conversation exchange.

In [5]:
# Instantiating and LLM object

llm = ChatOpenAI(model="gpt-4o-mini", api_key=secrets["api_keys"]["openai"])
llm.invoke("Test")

AIMessage(content='It looks like you’re testing the system. How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 8, 'total_tokens': 24, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_373a14eb6f', 'id': 'chatcmpl-DGJf3MLHW3iNkBG9c3cQvAKYKk0Ll', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019cc1fe-58fd-7da1-a9e1-8b67fbe0667d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 16, 'total_tokens': 24, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [6]:
res = llm.invoke(messages)
res

AIMessage(content='Sure! Here’s a brief overview:\n\n1. **MCP (Message Communication Protocol)**:\n   - **Purpose**: Used for communication between different systems or software components.\n   - **Example**: In an IoT application, MCP could enable a sensor to send temperature data to a central server.\n\n2. **A2A (Application-to-Application)**:\n   - **Purpose**: Facilitates communication and interaction between different applications, often in enterprise environments.\n   - **Example**: A CRM application using A2A to pull data from an inventory management system to update product availability in real-time.\n\nWhile both protocols handle communication, MCP is broader and often focuses on messages, whereas A2A is specifically about interactions between applications.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 149, 'prompt_tokens': 92, 'total_tokens': 241, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0,

In [7]:
res.pretty_print()

================================== Ai Message ==================================

Sure! Here’s a brief overview:

1. **MCP (Message Communication Protocol)**:
   - **Purpose**: Used for communication between different systems or software components.
   - **Example**: In an IoT application, MCP could enable a sensor to send temperature data to a central server.

2. **A2A (Application-to-Application)**:
   - **Purpose**: Facilitates communication and interaction between different applications, often in enterprise environments.
   - **Example**: A CRM application using A2A to pull data from an inventory management system to update product availability in real-time.

While both protocols handle communication, MCP is broader and often focuses on messages, whereas A2A is specifically about interactions between applications.


In [8]:
res.response_metadata

{'token_usage': {'completion_tokens': 149,
  'prompt_tokens': 92,
  'total_tokens': 241,
  'completion_tokens_details': {'accepted_prediction_tokens': 0,
   'audio_tokens': 0,
   'reasoning_tokens': 0,
   'rejected_prediction_tokens': 0},
  'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}},
 'model_provider': 'openai',
 'model_name': 'gpt-4o-mini-2024-07-18',
 'system_fingerprint': 'fp_373a14eb6f',
 'id': 'chatcmpl-DGJf7sXA47tKU90qJX3f3tXLT3myl',
 'service_tier': 'default',
 'finish_reason': 'stop',
 'logprobs': None}

# Part 3: Tools

Tools are essentially functions that can be exposed to an LLM for it to utlize. Based on the function description and the parameters needed by the function, the LLM can decide if it needs to utilize the tool to answer the users query.

In [9]:
# Creating a tool

def add_numbers(a: int, b: int) -> int:
    """Adds two numbers together."""
    return a + b

In [13]:
llm = llm.bind_tools([add_numbers])

In [15]:
add_res = llm.invoke([HumanMessage("What is 2 + 3?")])
add_res

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 54, 'total_tokens': 72, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_a1ddba3226', 'id': 'chatcmpl-DGK9QqkDtAxBDu4QuwA2HGhuNU5bd', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019cc21b-11a1-7791-bb3e-aa36b543bb3a-0', tool_calls=[{'name': 'add_numbers', 'args': {'a': 2, 'b': 3}, 'id': 'call_DJkGoRSm4XWOnEa8dZSNw3bk', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 54, 'output_tokens': 18, 'total_tokens': 72, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [16]:
add_res.tool_calls

[{'name': 'add_numbers',
  'args': {'a': 2, 'b': 3},
  'id': 'call_DJkGoRSm4XWOnEa8dZSNw3bk',
  'type': 'tool_call'}]

# Part 4: Tools in Graph Nodes